# 🚀 Rogii Geology Prediction: Elite Ensemble Pipeline
**Projeto Senior 09 - Fábrica de Ciência de Dados**

Este notebook foi desenhado para ser executado diretamente no Kaggle (Code Competition). Ele consolida todas as etapas de Engenharia de Dados (Filtro Savitzky-Golay, Lags Espaciais) e treina um Ensemble de Alta Performance (LightGBM + XGBoost) em tempo real.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from scipy.signal import savgol_filter
from sklearn.preprocessing import RobustScaler
import warnings
warnings.filterwarnings('ignore')

# 1. Configuração de Ambiente (Kaggle vs Local)
if os.path.exists('/kaggle/input'):
    print("Ambiente Kaggle detectado!")
    DATA_DIR = '/kaggle/input/rogii-wellbore-geology-prediction'
    OUTPUT_DIR = '/kaggle/working'
else:
    print("Ambiente Local detectado!")
    DATA_DIR = '../data/raw'
    OUTPUT_DIR = '.'

## 🛠️ Pipeline de Processamento de Sinais Geofísicos

In [ ]:
def process_well_data(file_path, is_train=True):
    """Processa um único poço aplicando filtros físicos e lag features."""
    df = pd.read_csv(file_path)
    well_id = os.path.basename(file_path).split('__')[0]
    df['well_id'] = well_id
    
    # Se for teste, guardar o row index original para a submissão
    if not is_train:
        df['row_idx'] = df.index
        
    # 1. Limpeza Física (Clipping e Interpolação)
    df['GR'] = df['GR'].clip(lower=0, upper=250)
    df['GR'] = df['GR'].bfill().ffill() # Proteção contra NaNs iniciais/finais
    
    # 2. Features Estruturais Rápidas
    df['GR_diff'] = df['GR'].diff().fillna(0)
    df['GR_rolling_mean'] = df['GR'].rolling(window=5, center=True).mean().fillna(df['GR'])
    df['Z_diff'] = df['Z'].diff().fillna(0)
    
    # 3. Geoprocessamento (Savitzky-Golay e Normalização)
    scaler = RobustScaler()
    if len(df) > 11:
        df['GR_savgol'] = savgol_filter(df['GR'], window_length=11, polyorder=3)
    else:
        df['GR_savgol'] = df['GR']
        
    df['GR_norm'] = scaler.fit_transform(df[['GR_savgol']])
    
    # 4. Lag Features (Memória Espacial)
    df = df.sort_values('MD')
    df['GR_lag_1'] = df['GR_norm'].shift(1).bfill()
    df['GR_lag_5'] = df['GR_norm'].shift(5).bfill()
    df['GR_delta_lag'] = df['GR_norm'] - df['GR_lag_1']
    
    return df

## 📥 Carga e Preparação em Larga Escala

In [ ]:
print("Processando dados de Treino...")
train_files = glob.glob(os.path.join(DATA_DIR, 'train', '*__horizontal_well.csv'))
train_dfs = [process_well_data(f, is_train=True) for f in train_files]
train_df = pd.concat(train_dfs, ignore_index=True)

print("Processando dados de Teste...")
test_files = glob.glob(os.path.join(DATA_DIR, 'test', '*__horizontal_well.csv'))
test_dfs = [process_well_data(f, is_train=False) for f in test_files]
test_df = pd.concat(test_dfs, ignore_index=True)

## 🤖 Treinamento do Ensemble Sênior

In [ ]:
FEATURES = [
    'GR_norm', 'GR_diff', 'GR_rolling_mean', 
    'GR_lag_1', 'GR_lag_5', 'GR_delta_lag',
    'Z', 'X', 'Y'
]
TARGET = 'TVT'

X_train = train_df[FEATURES]
y_train = train_df[TARGET]
X_test = test_df[FEATURES]

# Modelo 1: LightGBM Elite
print("Treinando LightGBM...")
model_lgb = lgb.LGBMRegressor(
    n_estimators=1000, learning_rate=0.03, num_leaves=127,
    max_depth=-1, min_child_samples=20, subsample=0.8,
    colsample_bytree=0.8, random_state=42, n_jobs=-1, verbose=-1
)
model_lgb.fit(X_train, y_train)
pred_lgb = model_lgb.predict(X_test)

# Modelo 2: XGBoost Robusto
print("Treinando XGBoost...")
model_xgb = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=8,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1
)
model_xgb.fit(X_train, y_train)
pred_xgb = model_xgb.predict(X_test)

## 🥇 Pós-Processamento e Submissão

In [ ]:
# Ensemble Blend (60% LGBM / 40% XGBoost)
test_df['pred_tvt'] = (pred_lgb * 0.6) + (pred_xgb * 0.4)

# Suavização final para consistência geológica (Rolling Mean)
processed_preds = []
for well_id, group in test_df.groupby('well_id'):
    group = group.copy().sort_values('MD')
    group['pred_tvt_smooth'] = group['pred_tvt'].rolling(window=3, center=True).mean().fillna(group['pred_tvt'])
    processed_preds.append(group)
    
test_df_final = pd.concat(processed_preds, ignore_index=True)

# Formatação Kaggle (<well_id>_<row_idx>)
test_df_final['kaggle_id'] = test_df_final['well_id'] + '_' + test_df_final['row_idx'].astype(str)

sub_template = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))
submission = pd.merge(sub_template[['id']], test_df_final[['kaggle_id', 'pred_tvt_smooth']], 
                      left_on='id', right_on='kaggle_id', how='left')

# Garantia contra NaNs
submission['pred_tvt_smooth'] = submission['pred_tvt_smooth'].fillna(0.0)
submission_final = submission[['id', 'pred_tvt_smooth']].rename(columns={'pred_tvt_smooth': 'tvt'})

submission_path = os.path.join(OUTPUT_DIR, 'submission.csv')
submission_final.to_csv(submission_path, index=False)
print(f"🎉 Arquivo final gerado com sucesso: {submission_path}")
print(submission_final.head())